# Chapter 13b — Atomics & Histograms (companion)

> Companion to **Chapter 13 — Block Reductions**, distilled from *CUDA by Example*
> (Sanders & Kandrot), **Chapter 9 — Atomics**.
> The other half of cross-block cooperation: combining results **on the GPU** without races.

In Chapter 13a the dot product reduced each block to one partial sum, then punted the final cross-block sum to the **CPU**. That's fine for 32 partials, but useless when thousands of blocks each want to add into the *same* location — exactly what happens when `encoder_backward` scatters token gradients into an embedding table, or when `global_norm` accumulates the sum of squares across the whole model.

The mechanism that makes this safe is the **atomic operation**: a read-modify-write that the hardware guarantees no other thread can interrupt.

### Learning objectives

By the end you will:

- Explain the **race condition** in `histo[x]++` and how `atomicAdd` fixes it.
- Build a 256-bin **histogram** two ways: naive global atomics vs. **shared-memory privatization**.
- Measure why naive global atomics are slow under **contention**, and how privatization cuts it.
- Connect atomics to `llm.c`'s `encoder_backward` (scatter-add) and `global_norm` (cross-block accumulate).


## 1. Concept — The Race in `histo[x]++`

A histogram counts how often each value appears: `for each x: histo[x]++`. On the GPU we'd love one thread per input element. But `histo[x]++` is **three** operations:

```
1. read  r = histo[x]
2. add   r = r + 1
3. write histo[x] = r
```

If two threads both have `x == 7` and run concurrently, both read the *old* value (say 40), both compute 41, both write 41. **Two increments, but the count only went up by one.** That's a *race condition* — the result depends on timing, and counts come out too low.

`atomicAdd` makes the read-modify-write **indivisible**: the hardware serializes concurrent atomics to the same address, so every increment lands.

```c
atomicAdd(&histo[x], 1);   // safe: no increment is ever lost
```

CUDA provides `atomicAdd`, `atomicSub`, `atomicMax`, `atomicCAS` (compare-and-swap, the universal primitive), etc., for both global and shared memory. `atomicAdd` on `float` exists too (used all over `llm.c`), though float atomics aren't associative, so results vary slightly run-to-run.


## 2. Demo — Histogram Two Ways

We'll histogram a large buffer of random bytes (values 0–255) into 256 bins, and verify the GPU counts match a CPU reference **exactly** (integer atomics are lossless). Two kernels:

- **`histo_global`**: every thread does `atomicAdd(&histo[buf[i]], 1)` straight into global memory. Correct, but with only 256 bins and millions of elements, thousands of threads hammer the same 256 addresses — brutal contention.
- **`histo_shared`**: each block keeps a **private** 256-bin histogram in shared memory, atomically updates *that* (fast on-chip atomics, contention limited to one block), then does **256** global atomics at the end to merge. Far less global contention.


In [ ]:
!mkdir -p course/ch13b_build


In [ ]:
%%writefile course/ch13b_build/histogram.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define SIZE   (64 * 1024 * 1024)   // 64M bytes
#define NBINS  256

// naive: atomics straight into global memory — heavy contention on 256 addresses
__global__ void histo_global(const unsigned char* buf, long size, unsigned int* histo) {
    int i      = threadIdx.x + blockIdx.x * blockDim.x;
    int stride = blockDim.x * gridDim.x;
    while (i < size) { atomicAdd(&histo[buf[i]], 1); i += stride; }
}

// privatized: per-block histogram in shared memory, merged once at the end
__global__ void histo_shared(const unsigned char* buf, long size, unsigned int* histo) {
    __shared__ unsigned int temp[NBINS];     // launch with blockDim.x == NBINS
    temp[threadIdx.x] = 0;
    __syncthreads();

    int i      = threadIdx.x + blockIdx.x * blockDim.x;
    int stride = blockDim.x * gridDim.x;
    while (i < size) { atomicAdd(&temp[buf[i]], 1); i += stride; }   // on-chip atomics
    __syncthreads();

    atomicAdd(&histo[threadIdx.x], temp[threadIdx.x]);              // 256 global atomics / block
}

int main(void) {
    unsigned char* buf = (unsigned char*)malloc(SIZE);
    srand(1234);
    unsigned int cpu[NBINS] = {0};
    for (long i = 0; i < SIZE; i++) { buf[i] = rand() & 0xff; cpu[buf[i]]++; }

    unsigned char* d_buf; unsigned int* d_histo;
    cudaMalloc(&d_buf, SIZE); cudaMalloc(&d_histo, NBINS * sizeof(unsigned int));
    cudaMemcpy(d_buf, buf, SIZE, cudaMemcpyHostToDevice);

    cudaEvent_t s, e; cudaEventCreate(&s); cudaEventCreate(&e);
    unsigned int out[NBINS];
    int block = 256;
    int grid  = 1024;

    // --- global-atomic version ---
    cudaMemset(d_histo, 0, NBINS * sizeof(unsigned int));
    cudaEventRecord(s);
    histo_global<<<grid, block>>>(d_buf, SIZE, d_histo);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms_g; cudaEventElapsedTime(&ms_g, s, e);
    cudaMemcpy(out, d_histo, sizeof(out), cudaMemcpyDeviceToHost);
    int ok_g = 1; for (int b = 0; b < NBINS; b++) if (out[b] != cpu[b]) ok_g = 0;

    // --- shared-memory privatized version (blockDim MUST be NBINS) ---
    cudaMemset(d_histo, 0, NBINS * sizeof(unsigned int));
    cudaEventRecord(s);
    histo_shared<<<grid, NBINS>>>(d_buf, SIZE, d_histo);
    cudaEventRecord(e); cudaEventSynchronize(e);
    float ms_s; cudaEventElapsedTime(&ms_s, s, e);
    cudaMemcpy(out, d_histo, sizeof(out), cudaMemcpyDeviceToHost);
    int ok_s = 1; long total = 0;
    for (int b = 0; b < NBINS; b++) { if (out[b] != cpu[b]) ok_s = 0; total += out[b]; }

    printf("global atomics : %6.3f ms   counts %s\n", ms_g, ok_g ? "match CPU" : "WRONG");
    printf("shared privat. : %6.3f ms   counts %s\n", ms_s, ok_s ? "match CPU" : "WRONG");
    printf("speedup        : %.2fx     (total counted = %ld, expected %d)\n",
           ms_g / ms_s, total, SIZE);

    cudaFree(d_buf); cudaFree(d_histo); free(buf);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13b_build/histogram course/ch13b_build/histogram.cu && ./course/ch13b_build/histogram


Both versions produce counts that match the CPU **exactly** (integer atomics lose nothing). The shared-memory version is *dramatically* faster — on this RTX 4080 SUPER it comes out ~200× faster (e.g. ~22 ms → ~0.1 ms). The reason: it replaces millions of *global* atomics contending on just 256 addresses with millions of *shared* atomics (on-chip, per-block) plus only `256 × gridDim` global atomics at the end. The exact factor depends on the GPU and how badly the bins collide, but the direction is always the same and the gap is large.

The lesson generalizes: **atomics are correct but contended atomics are slow.** Push the contention down to the smallest, fastest scope you can — a register (a plain reduction, Chapter 13a), then shared memory, and only then global.


## 3. When to Use Atomics vs. a Reduction

| Situation | Best tool |
|---|---|
| Sum a big array to one scalar | **Reduction** (Chapter 13a) — no atomics needed in the hot loop |
| Many threads update *few* shared slots (histogram, embedding grads) | **Atomics**, privatized in shared memory |
| Combine a handful of per-block partials | One global `atomicAdd` per block (cheap — few contenders) |
| Need a lock / custom update | `atomicCAS` loop (Appendix03) |

Rule of thumb: if a reduction can express it, prefer the reduction — it touches each address from exactly one thread. Reach for atomics when the *write targets collide unpredictably* (you don't know at compile time which bin/row each thread hits), which is exactly the embedding-gradient case below.


## 4. Translation Bridge — Atomics in `llm.c`

| Histogram (book Ch9) | `llm.c` | Why atomics |
|---|---|---|
| `atomicAdd(&histo[buf[i]], 1)` | `encoder_backward`: `atomicAdd(&dwte[token*C + c], grad)` | many tokens in a batch map to the **same** embedding row — collisions unknown until runtime |
| per-block `__shared__ temp[256]` | warp/block-level grad staging before the global scatter | cut global-atomic contention |
| `atomicAdd(&histo[tid], temp[tid])` final merge | `global_norm`: `atomicAdd(out, block_partial)` | combine one partial per block into the grand total on-GPU |
| integer counts (exact) | **float** grad atomics (slightly nondeterministic) | float add isn't associative — repeated runs differ in the last bits |

This is why `llm.c` training is not bit-for-bit reproducible across runs even on the same GPU: the float `atomicAdd`s in the backward pass commit in nondeterministic order. It's expected and harmless — the differences are at the rounding-error level.


## 5. Common Pitfalls

- **`histo[x]++` without an atomic** → silently undercounts. The kernel *runs* and looks plausible; only the totals are wrong.
- **`histo_shared` requires `blockDim.x == NBINS`** here, because each thread zeroes and flushes exactly one bin (`temp[threadIdx.x]`). Change the bin count and you must change the launch or the indexing.
- **Float atomics are nondeterministic** — never assert bit-exact equality across runs for float `atomicAdd` results.
- **Over-using atomics**: a global `atomicAdd` per element where a reduction would do is a classic performance bug.
- **`atomicAdd` on `double`** needs compute capability ≥ 6.0; on the 4080 (8.9) it's fine, but older targets may need an `atomicCAS` fallback.


## 6. TODO Exercise — Atomic Scatter-Add (mini `encoder_backward`)

Each of `N` tokens has an index in `[0, NROWS)` and a scalar gradient. Accumulate each token's gradient into its row. Multiple tokens share rows, so the naive `grad_table[row] += g` races — fix it with an atomic. Fill in the one TODO.


In [ ]:
%%writefile course/ch13b_build/exercise1.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N      (1 << 20)
#define NROWS  64

__global__ void scatter_add(const int* idx, const float* grad, float* table, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    int stride = blockDim.x * gridDim.x;
    while (i < n) {
        // TODO: atomically add grad[i] into table[idx[i]]
        // (replace the racy line below with an atomicAdd)
        table[idx[i]] += grad[i];
        i += stride;
    }
}

int main(void) {
    int*   idx  = (int*)  malloc(N*sizeof(int));
    float* grad = (float*)malloc(N*sizeof(float));
    double ref[NROWS] = {0};
    srand(7);
    for (int i = 0; i < N; i++) { idx[i] = rand() % NROWS; grad[i] = 1.0f; ref[idx[i]] += 1.0; }

    int *d_idx; float *d_grad, *d_table;
    cudaMalloc(&d_idx, N*sizeof(int)); cudaMalloc(&d_grad, N*sizeof(float));
    cudaMalloc(&d_table, NROWS*sizeof(float));
    cudaMemcpy(d_idx, idx, N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_grad, grad, N*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemset(d_table, 0, NROWS*sizeof(float));

    scatter_add<<<256, 256>>>(d_idx, d_grad, d_table, N);
    float table[NROWS]; cudaMemcpy(table, d_table, sizeof(table), cudaMemcpyDeviceToHost);

    int ok = 1; for (int r = 0; r < NROWS; r++) if (fabs(table[r] - ref[r]) > 0.5) ok = 0;
    printf("row0 gpu=%.0f ref=%.0f  -> %s\n", table[0], ref[0], ok ? "PASS" : "FAIL");
    free(idx); free(grad); cudaFree(d_idx); cudaFree(d_grad); cudaFree(d_table);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13b_build/exercise1 course/ch13b_build/exercise1.cu && ./course/ch13b_build/exercise1


### Solution

In [ ]:
%%writefile course/ch13b_build/exercise1_sol.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define N      (1 << 20)
#define NROWS  64

__global__ void scatter_add(const int* idx, const float* grad, float* table, int n) {
    int i = threadIdx.x + blockIdx.x * blockDim.x;
    int stride = blockDim.x * gridDim.x;
    while (i < n) {
        atomicAdd(&table[idx[i]], grad[i]);   // collisions on rows are unknown at compile time
        i += stride;
    }
}

int main(void) {
    int*   idx  = (int*)  malloc(N*sizeof(int));
    float* grad = (float*)malloc(N*sizeof(float));
    double ref[NROWS] = {0};
    srand(7);
    for (int i = 0; i < N; i++) { idx[i] = rand() % NROWS; grad[i] = 1.0f; ref[idx[i]] += 1.0; }

    int *d_idx; float *d_grad, *d_table;
    cudaMalloc(&d_idx, N*sizeof(int)); cudaMalloc(&d_grad, N*sizeof(float));
    cudaMalloc(&d_table, NROWS*sizeof(float));
    cudaMemcpy(d_idx, idx, N*sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_grad, grad, N*sizeof(float), cudaMemcpyHostToDevice);
    cudaMemset(d_table, 0, NROWS*sizeof(float));

    scatter_add<<<256, 256>>>(d_idx, d_grad, d_table, N);
    float table[NROWS]; cudaMemcpy(table, d_table, sizeof(table), cudaMemcpyDeviceToHost);

    int ok = 1; for (int r = 0; r < NROWS; r++) if (fabs(table[r] - ref[r]) > 0.5) ok = 0;
    printf("row0 gpu=%.0f ref=%.0f  -> %s\n", table[0], ref[0], ok ? "PASS" : "FAIL");
    free(idx); free(grad); cudaFree(d_idx); cudaFree(d_grad); cudaFree(d_table);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch13b_build/exercise1_sol course/ch13b_build/exercise1_sol.cu && ./course/ch13b_build/exercise1_sol


## Recap

- An **atomic** is an indivisible read-modify-write; it fixes the race in `histo[x]++` / `table[row] += g`.
- **Contended global atomics are slow.** Privatize into shared memory first (per-block histogram), then merge with a few global atomics.
- Prefer a **reduction** when one thread can own each output; reach for **atomics** when write targets collide unpredictably (embedding gradients).
- `llm.c` uses float atomics in `encoder_backward` (scatter-add) and `global_norm` (cross-block sum) — which is exactly why training isn't bit-reproducible.

### What's next

You've now seen both ways blocks cooperate beyond their own boundary: **reductions** (13a) and **atomics** (13b). Next in the course proper is **Chapter 14 — cuBLAS**, where NVIDIA's library does the matmul reductions for you. The independent appendix companions (constant memory & events, streams) round out the book's toolkit.
